In [43]:
import io
from pathlib import Path

from PIL import Image
from rembg import new_session, remove


session = new_session("u2netp", providers=["CPUExecutionProvider"])#force cpu mode fro avoiding gpu memory issues

base_dir = Path(r"D:\downloads\archive (1)\Herbify-Dataset")#to  get folder from desktop
input_folder = base_dir / "Globe Amarnath"
output_folder = base_dir / "without background class" / "Globe Amarnath"
target_size = (256, 256)

if not input_folder.exists():
    raise FileNotFoundError(f"Input folder does not exist: {input_folder}")


output_folder.mkdir(parents=True, exist_ok=True)#to creat folder if not exist


def process_image(file_path: Path, output_path: Path):
    with Image.open(file_path) as img:
        result = remove(img, session=session)

        if isinstance(result, bytes):
            img_no_bg = Image.open(io.BytesIO(result)).convert("RGBA")
        elif isinstance(result, Image.Image):
            img_no_bg = result.convert("RGBA")
        else:
            raise TypeError("rembg returned an unsupported type.")

        if hasattr(Image, "Resampling"):
            img_resized = img_no_bg.resize(target_size, Image.Resampling.LANCZOS)
        else:
            img_resized = img_no_bg.resize(target_size, Image.ANTIALIAS)

        img_resized.save(output_path.with_suffix(".png"), format="PNG")


for file_path in sorted(input_folder.iterdir()):
    if file_path.is_file() and file_path.suffix.lower() in {".png", ".jpg", ".jpeg"}:
        process_image(file_path, output_folder / file_path.name)

print(f"✅ All images processed and saved to: {output_folder}")


✅ All images processed and saved to: D:\downloads\archive (1)\Herbify-Dataset\without background class\Globe Amarnath
